# 5-2 · 어느 점수부터 APS로 판정할까 — 검증에서 고르고 테스트에서 확인

**⏱ 60분** · 5주차 교재 2/2 — (5-1) 두 오류와 비용, 자료 나누기, 모델 점수 → 검증에서 판정 기준을 고르고 테스트에서 확인하기

## 🎯 이번 시간의 질문

5-1에서 검증 기록 12,000건에 모델 점수 `p_val`을 매겼습니다. 모두 "APS 아님"으로 판정하면 비용이 100,000, 모두 "APS"로 판정하면 118,000이었습니다.

> **오경보 비용 10, 미탐지 비용 500일 때 어느 점수부터 APS 고장으로 판정할까요? 그 기준은 흔히 쓰는 0.5보다 비용을 줄일까요?**

**기준**(임계값)은 점수가 이 값 이상이면 APS로 판정하는 경계입니다. 기준을 낮추면 APS 판정이 늘어 미탐지는 줄고 오경보는 늘어납니다. 검증 자료에서 가능한 기준을 모두 비교해 비용이 가장 낮은 기준을 고르고, 그 기준을 고정한 채 테스트 파일에서 평가합니다.

혼동행렬·기대비용·확률 보정·모델 설정의 뜻은 **[5주차 배경지식 — 판정과 비용](https://balab-pknu.github.io/bizanalytics/theory/week05.html)** 문서에 있습니다.

**이번 시간에 반복할 패턴:** 판정 `(점수 >= 기준).astype(int)` → `errors(정답, 판정)` → 비용 → 후보마다 반복해 가장 싼 기준 고르기

셀 표시는 5-1과 같습니다: ▶ 그대로 실행 · ✍️ 보고 타이핑 · 📝 과제 · 💬 결과 해설 · ✅ 체크포인트

> **실습본입니다.** 먼저 메뉴 **파일 → Drive에 사본 저장**을 누르세요. 사본을 저장하지 않으면 입력한 코드가 사라집니다.
> 맨 위 준비 셀부터 순서대로 `Shift + Enter`로 실행합니다. ▶ 셀은 그대로 실행하고, ✍️ 셀은 위에 보여 준 코드를
> 빈 셀에 손으로 타이핑하고, 빈칸(`___`)은 채워서 실행합니다. 📝 과제는 힌트를 보고 스스로 풉니다.

In [ ]:
# ▶ 준비 셀 — 그대로 실행하세요. 5-1과 같은 분할·중앙값·모델 설정으로 검증 점수 p_val을 다시 만듭니다. (학습 30초~1분)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier
import os
import sys

for folder in (".", "..", "../..", "../../.."):
    if os.path.exists(os.path.join(folder, "balab.py")):
        sys.path.insert(0, folder)
        break
else:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/BALAB-PKNU/bizanalytics/main/balab.py", "balab.py")

from balab import load
aps = load("aps")
tr = aps["train"]
te = aps["test"]
X = tr.drop(columns=["class"])
y = (tr["class"] == "pos").astype(int)
X_fit, X_val, y_fit, y_val = train_test_split(
    X, y, test_size=12000, stratify=y, random_state=0)
med = X_fit.median()
X_fit = X_fit.fillna(med)
X_val = X_val.fillna(med)
model = LGBMClassifier(
    num_leaves=15, learning_rate=0.03, n_estimators=800,
    min_child_samples=20, subsample=0.8, subsample_freq=1,
    colsample_bytree=0.8, verbose=-1, random_state=0,
    n_jobs=1, deterministic=True, force_col_wise=True)
model.fit(X_fit, y_fit)
p_val = pd.Series(model.predict_proba(X_val)[:, 1], index=X_val.index)
print("준비 완료!", len(X_fit), len(X_val), round(p_val.median(), 7))

✅ `준비 완료! 48000 12000 1.49e-05`가 나오면 됩니다. 5-1과 같은 점수입니다.

---
## 1. 두 오류를 세는 함수 다시 만들기

5-1 2절의 `errors` 함수를 다시 만듭니다. 이번 시간 내내 씁니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
def errors(actual, predicted):
    fp = int(((predicted == 1) & (actual == 0)).sum())
    fn = int(((predicted == 0) & (actual == 1)).sum())
    return fp, fn
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


---
## 2. 비용이 같은 기준이 여럿이면?

기준을 조금 바꿔도 판정이 같으면 비용도 같습니다. 가장 싼 기준이 여럿일 때 어느 것을 고를지 미리 정해 둡니다. 이 장에서는 **가장 큰 기준**을 고릅니다. 기준이 클수록 APS로 판정하는 기록이 적거나 같기 때문입니다.

후보 세 개짜리 연습 표로 먼저 해 봅니다.

In [ ]:
# ▶ 그대로 실행하세요 — 연습용 후보 표 (기준 오름차순)
example = pd.DataFrame({"threshold": [0.1, 0.2, 0.3], "cost": [30, 20, 20]})
example

**가장 싼 행 고르기** — `.min()`으로 최소 비용을 구하고, `.loc[조건]`으로 비용이 최소인 행만 남긴 뒤, `.iloc[-1]`로 마지막 행을 고릅니다. 표가 기준 오름차순이므로 마지막 행이 가장 큰 기준입니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
example.loc[example["cost"] == example["cost"].min()].iloc[-1]
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 비용 20인 두 행(0.2, 0.3) 중 **0.3**을 골랐습니다.

---
## 3. 검증 자료에서 모든 기준을 비교한다

### 후보 모으기

판정은 기준이 관측된 점수를 지날 때만 바뀝니다. 그래서 검증 점수의 서로 다른 값을 모두 후보로 쓰면, 그 사이에 어떤 기준을 넣어도 새 판정이 생기지 않습니다. 앞에는 전원 "APS" 판정인 0, 뒤에는 전원 "APS 아님" 판정인 무한대(`float("inf")`)를 붙입니다. `.unique()`는 중복을 없애고 `sorted`는 오름차순으로 놓습니다. 리스트는 `+`로 이어 붙입니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
thresholds = [0.0] + sorted(p_val.unique()) + [float("inf")]
len(thresholds)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 후보가 11,720개입니다.

### 후보마다 두 오류와 비용 계산

이번 주의 핵심 반복입니다. 후보를 하나씩 꺼내(`for`) **판정 → 두 오류 → 기록**을 되풀이하고, 모은 결과(`rows`)를 표로 만든 뒤 비용 열을 붙입니다. `positive`는 APS로 판정한 기록 수입니다. 15초쯤 걸립니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
rows = []
for threshold in thresholds:
    predicted = (p_val >= threshold).astype(int)
    fp, fn = errors(y_val, predicted)
    rows.append([threshold, fp, fn, int(predicted.sum())])
curve = pd.DataFrame(rows, columns=["threshold", "fp", "fn", "positive"])
curve["cost"] = 10 * curve["fp"] + 500 * curve["fn"]
curve.head()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 첫 행(기준 0)은 전원 "APS" 판정이라 오경보 11,800건, 비용 118,000입니다. 5-1 3절의 값과 같습니다. 기준이 올라갈수록 오경보가 한 건씩 줄어듭니다.

### 가장 싼 기준 고르기

2절과 같은 방법으로 비용이 가장 낮은 행을 고르고, 그 기준을 `t_star`에 저장합니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
best = curve.loc[curve["cost"] == curve["cost"].min()].iloc[-1]
t_star = float(best["threshold"])
best
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — `threshold 0.014902`, `fp 186`, `fn 6`, `cost 4860`이면 맞습니다.

💬 **결과 해설**
- 검증에서 고른 기준은 약 **0.0149**입니다. 기본값 0.5보다 훨씬 낮습니다.
- 오경보 186건(비용 1,860) + 미탐지 6건(비용 3,000) = **4,860**입니다. APS로 판정한 기록은 380건입니다.
- 미탐지 한 건이 오경보 50건과 같은 값이라, 오경보를 꽤 늘리더라도 미탐지를 줄이는 쪽이 싸게 나왔습니다.
- 이 비용은 기준을 고른 바로 그 자료의 결과입니다. 처음 보는 기록에서의 비용은 6절에서 테스트로 확인합니다.

---
## 4. 기준에 따라 비용은 어떻게 변하나

모든 후보의 비용을 그림으로 봅니다. 점수가 0 근처에 몰려 있어 가로축은 로그 눈금입니다. 빨간 선이 검증에서 고른 기준, 회색 점선은 **비용식의 기준** 10/(10+500) ≈ 0.0196입니다. 점수가 실제 확률과 정확히 맞는다면 APS 확률이 이 값 이상일 때 APS로 판정하는 것이 기대비용이 낮습니다(배경지식 2절).

In [ ]:
# ▶ 그대로 실행하세요 — 검증 비용 곡선 (0과 무한대는 로그 축에 그릴 수 없어 뺍니다)
plot_curve = curve.loc[(curve["threshold"] > 0) & (curve["threshold"] < float("inf"))]
plt.figure(figsize=(8, 4))
plt.plot(plot_curve["threshold"], plot_curve["cost"])
plt.xscale("log")
plt.axvline(t_star, color="red", label="Selected on validation")
plt.axvline(10 / 510, color="gray", linestyle="--", label="Cost formula")
plt.xlabel("Threshold")
plt.ylabel("Validation cost")
plt.legend()
plt.show()

💬 **결과 해설**
- 기준이 아주 낮으면 오경보가 쌓여 비용이 크고, 높으면 미탐지가 쌓여 비용이 큽니다. 그 사이의 낮은 골짜기에서 기준을 골랐습니다.
- 검증에서 고른 기준(0.0149)과 비용식의 기준(0.0196)은 가깝지만 같지 않습니다. 점수가 실제 확률과 딱 맞지 않을 수도 있고, 검증 표본이 12,000건뿐이라 생기는 흔들림도 있습니다. 어느 쪽 때문인지는 이 그림만으로 정하지 않습니다.

---
## 5. 입력 하나만 쓰는 규칙과 비교한다

모델이 정말 필요한지 보려면 더 단순한 규칙과 비교해야 합니다. 입력 열 하나 `aa_000`만 보고 "이 값 이상이면 APS"로 판정하는 규칙도 **같은 검증 자료, 같은 비용**으로 기준을 고릅니다. `aa_000`은 비교를 위해 미리 정한 열이며, 모든 열 중 가장 좋은 열이라는 뜻은 아닙니다.

3절의 반복문과 같은 구조입니다. 이번에는 빈칸을 채워 직접 완성합니다. 후보는 `aa_000`의 서로 다른 값에 음의 무한대(전원 "APS")와 무한대(전원 "APS 아님")를 붙였습니다.

In [ ]:
# ✍️ 빈칸(___)을 채운 뒤 실행하세요
cuts = [float("-inf")] + sorted(X_val["aa_000"].unique()) + [float("inf")]
rows = []
for cut in cuts:
    predicted = (___ >= cut).astype(int)
    fp, fn = errors(y_val, predicted)
    rows.append([cut, fp, fn])
rule_curve = pd.DataFrame(rows, columns=["threshold", "fp", "fn"])
rule_curve["cost"] = ___
rule_curve.head()

3절처럼 가장 싼 행을 고르고(동률이면 가장 큰 기준), 그 기준을 `best_cut`에 저장합니다.

In [ ]:
# ✍️ 빈칸(___)을 채운 뒤 실행하세요
rule_best = ___
best_cut = float(rule_best["threshold"])
rule_best

✅ **체크포인트** — `threshold 136740`, `fp 785`, `fn 11`, `cost 13350`이면 맞습니다.

💬 입력 하나만 쓰는 규칙의 가장 싼 검증 비용은 **13,350**으로, 모델 기준(4,860)의 약 2.7배입니다. 두 극단(100,000·118,000)보다는 훨씬 낮습니다. 모델 기준과 이 규칙의 기준은 여기서 고정하고, 테스트 결과를 본 뒤 다시 고르지 않습니다.

---
## 6. 테스트에서 고정한 규칙을 비교한다

이제 선택에 쓰지 않은 테스트 파일 16,000건에서 평가합니다. 테스트 입력의 빈 값도 **학습용 중앙값** `med`로 채우고, 모델은 다시 학습하지 않습니다. 5-1 4절과 같은 방법으로 점수 `p_te`를 매깁니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
X_te = te.drop(columns=["class"]).fillna(med)
y_te = (te["class"] == "pos").astype(int)
p_te = pd.Series(model.predict_proba(X_te)[:, 1], index=X_te.index)
y_te.value_counts()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 테스트에는 다른 고장 15,625건, APS 375건이 있습니다.

비교할 규칙 여섯 개의 테스트 판정을 만듭니다. 두 극단, 입력 하나 규칙, 기본값 0.5, 비용식 기준, 검증에서 고른 기준입니다. 규칙 이름과 판정을 짝지어 **사전**(`{이름: 판정}`)에 담는 코드는 실행만 합니다.

In [ ]:
# ▶ 그대로 실행하세요 — 규칙별 테스트 판정
rules = {"none": y_te * 0,
         "all": y_te * 0 + 1,
         "single_variable": (X_te["aa_000"] >= best_cut).astype(int),
         "model_0.5": (p_te >= 0.5).astype(int),
         "model_formula": (p_te >= 10 / 510).astype(int),
         "model_selected": (p_te >= t_star).astype(int)}

규칙마다 **두 오류 → 비용**을 계산해 한 표로 모읍니다. 3절의 반복과 같은 구조이고, `.items()`는 사전에서 이름과 판정을 한 쌍씩 꺼냅니다. 정확도(판정과 정답이 같은 비율)도 함께 봅니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
rows = []
for name, predicted in rules.items():
    fp, fn = errors(y_te, predicted)
    rows.append([name, fp, fn, int(predicted.sum()), 10 * fp + 500 * fn, (predicted == y_te).mean()])
result = pd.DataFrame(rows, columns=["rule", "fp", "fn", "positive", "cost", "accuracy"]).set_index("rule")
result
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — `model_selected`의 cost가 14340, `model_0.5`의 cost가 41230이면 맞습니다.

💬 **결과 해설** — 기본값 0.5와 검증에서 고른 기준을 비교해 봅시다.

| 규칙 | 오경보 | 미탐지 | 비용 | 정확도 |
|---|---|---|---|---|
| 기본값 0.5 | 23 | 82 | 41,230 | 0.993 |
| 검증에서 고른 기준 | 234 | 24 | **14,340** | 0.984 |

- 미탐지가 82건에서 24건으로 줄고 오경보는 23건에서 234건으로 늘었습니다. 미탐지 58건 × 500이 오경보 211건 × 10보다 훨씬 커서 총비용이 크게 낮아졌습니다.
- 정확도는 기본값 0.5가 더 높습니다. 정확도가 높다고 비용이 낮은 것은 아닙니다.
- 입력 하나 규칙(21,460)과 비용식 기준(16,050)보다도 검증에서 고른 기준이 낮았습니다. 전원 "APS 아님"은 정확도 0.977인데 비용은 187,500입니다.

기본값 대비 비용이 몇 배인지 계산합니다. `result.loc[행, 열]`로 두 비용을 꺼내 나눕니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
result.loc["model_selected", "cost"] / result.loc["model_0.5", "cost"]
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — `0.3478...`이면 맞습니다.

💬 검증에서 고른 기준의 테스트 비용은 기본값 0.5의 **약 0.35배**입니다.

검증 비용(4,860)과 테스트 비용(14,340)이 다른 이유를 "검증에 맞춰 골라서"라는 하나로 단정하지 않습니다. 두 자료는 건수(12,000·16,000)와 APS 수(200·375)도 다릅니다. 이 표를 본 뒤 기준이나 모델 설정을 바꿔 더 좋은 값을 찾지 않습니다.

---
## 7. 결정과 해석의 범위

> **오경보 10, 미탐지 500의 비용에서는 기본값 0.5 대신 검증에서 고른 기준 0.0149 이상을 APS로 판정한다.** 테스트에서 비용이 41,230에서 14,340으로(약 0.35배) 줄었다.

- 기본값 0.5는 두 오류를 같은 가격으로 볼 때의 기준입니다. 이 문제처럼 미탐지가 50배 비싸면 정확도나 0.5로 규칙을 고르면 목적과 어긋납니다.
- 0.0149는 이 모델·전처리·표본·비용표에 맞춘 값입니다. 비용만 바뀌면 같은 검증 점수의 `curve`에서 비용 열만 다시 계산해 고를 수 있습니다(랩). 자료의 분포나 입력 정의까지 달라지면 모델과 검증을 다시 확인해야 합니다.
- 이 오류 수는 APS 고장과 다른 고장을 가려낸 결과입니다. 줄어든 미탐지를 실제로 예방한 고장 수로 읽지 않습니다.

---
## 8. 실습 과제 (10분)

### 📝 과제 1 · 비용식 기준은 검증에서 얼마였나

4절의 회색 점선, 비용식 기준 10/510으로 검증 자료를 판정하면 오경보·미탐지·비용은 얼마인가요? 검증에서 고른 기준(4,860)과 비교하세요.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: 3절 반복문 안의 판정 한 줄과 errors 한 줄을 기준 10 / 510으로 한 번만 쓰고, 두 오류로 비용을 계산합니다.


✏️ **나의 답:** (여기를 두 번 눌러 적습니다)

### 📝 과제 2 · 정확도가 가장 높은 기준을 고르면?

3절의 `curve`에 검증 정확도 열을 붙이고, 정확도가 가장 높은 행을 찾으세요. 그 행의 기준·오경보·미탐지·비용을 검증에서 고른 기준과 비교합니다.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: 정확도는 1에서 (오경보 + 미탐지) / 검증 건수를 뺀 값입니다. curve에 accuracy 열을 만들고, 2절의 가장 싼 행 고르기에서 cost와 min을 accuracy와 max로 바꿉니다.


✏️ **나의 답:**

### 📝 과제 3 · 생각해 보기

- 테스트에서 비용이 줄었다면 오경보와 미탐지 중 무엇이 얼마나 달라졌나요?
- 정확도가 높아도 이 비용표에서 불리한 규칙은 무엇인가요?
- 6절의 표만으로 "고장을 미리 막았다"고 말할 수 없는 이유는 무엇인가요?

✏️ **나의 답:**

---
## ✅ 정리

| 배운 것 | 핵심 |
|---|---|
| 판정 기준 | 점수가 기준 이상이면 APS. 기준을 낮추면 미탐지 ↓ 오경보 ↑ |
| 후보 | 검증 점수의 서로 다른 값 전부 + 0과 무한대(11,720개). 비용이 같으면 가장 큰 기준 |
| 검증에서 고른 기준 | 약 0.0149 — 오경보 186, 미탐지 6, 비용 4,860 (입력 하나 규칙은 13,350) |
| 테스트 | 선택 기준 14,340 · 비용식 기준 16,050 · 입력 하나 21,460 · 기본값 0.5 41,230 → 기본값의 약 0.35배 |
| 정확도와 비용 | 정확도가 높은 규칙(0.5, 전원 "APS 아님")이 비용은 크다 |
| 해석의 범위 | 이 모델·표본·비용표에 맞춘 기준. 줄어든 미탐지를 예방한 고장으로 읽지 않는다 |
| 반복한 패턴 | `(점수 >= 기준).astype(int)` → `errors(정답, 판정)` → `10 * fp + 500 * fn` → 가장 싼 행 |

### 이번 주에 할 일

| 활동 | 자료 | 비고 |
|---|---|---|
| 배경지식 | [판정과 비용](https://balab-pknu.github.io/bizanalytics/theory/week05.html) | 두 오류, 비용에서 기준 도출, 자료의 역할, 확률 보정 |
| 교재 | 5-1, 5-2 노트북 | 두 오류를 비용으로 세고, 검증에서 기준을 골라 테스트에서 확인한다 |
| 랩 | `week05_lab.ipynb` — 같은 APS 데이터 | 오류의 비용이 바뀌면 판정 기준은 어떻게 달라질까? 약 60분 |